# PC-FNO3D NATIONAL — OS7 : prévision du vent 3D à +24 h et +48 h (Colab CPU puis A100)

**NE PAS LANCER sans l'accord écrit d'Ines.** La règle `configs/national/regle_os7_national.json`
(sha256 `4360860450c4…`, fixée le 2026-11-11) est une **PROPOSITION soumise à l'accord écrit
d'Ines** ; le carnet vérifie son empreinte et celle de la règle v4, et refuse de tourner tant que
`ACCORD_ECRIT_INES` est vide. Code : dépôt public au commit `d3b2f07260a6` (à pousser avant de lancer).

Le modèle national **v4 choisi** (`runs/choix_v4.json`, choisi sur la validation), nourri par la **prévision
GFS** (NOAA, AWS, `noaa-gfs-bdp-pds`, GRIB2 0,25°, lecture par plages d'octets via `.idx`) au lieu d'ERA5 :
mêmes mailles, mêmes hauteurs au-dessus du sol et mêmes formules que le forçage ERA5 national, aucune
station en entrée. Vérité d'entraînement v3 et découpage inchangés.

| Partie | Environnement | Contenu |
|---|---|---|
| A | **Colab CPU** (mémoire élevée) | GFS f24 et f48 aux instants du cache v3 (boîte nationale ; les échéances de l'OS7 du Grand Tunis sont relues sans y écrire) → `MyDrive/pcfno3d_national/os7/forcage_gfs_f<L>.csv/.npz` ; environ 6 à 8 h, reprise par instant |
| B | **Colab A100** | référence sans affinage ; affinage **v5** (12 époques, GFS f24, époque choisie sur la validation, `runs/national_1km_v5_gfs`) ; **PREV1** (+24 h), **PREV2** (+48 h) aux 12 aéroports METAR contre GFS brut et la persistance ; **PREV3** (décision de traitement de la veille) ; champ 3D à +24 h contre la vérité d'entraînement (information) ; verdicts par `scripts/juger_os7_national.py` |

Un seul **Tout exécuter** par environnement : le carnet détecte le sien. Le test 2022 n'est lu qu'une fois
par modèle et par échéance (une étape dont la sortie existe est sautée). La session s'arrête à la fin de
chaque partie. GFS et la vérité générée sont des prévisions et une vérité d'entraînement, jamais des
mesures ; les mesures sont les METAR.

## 1. Accord, Drive, code au commit figé, données

In [ ]:
# ACCORD ECRIT D'INES : la regle configs/national/regle_os7_national.json est une PROPOSITION. Ce carnet ne
# telecharge, n'affine et ne lit le test qu'une fois l'accord ecrit recu ; le recopier ici (date et termes).
ACCORD_ECRIT_INES = ""
if not ACCORD_ECRIT_INES.strip():
    raise SystemExit("accord ecrit d'Ines absent : rien n'est lance (remplir ACCORD_ECRIT_INES)")
print("accord :", ACCORD_ECRIT_INES)

from google.colab import drive
import os as _os
if not _os.path.isdir("/content/drive/MyDrive"):   # deja monte (panneau Fichiers) : ne pas redemander
    drive.mount("/content/drive")

import json, os, shutil, subprocess, sys, time, zipfile
from pathlib import Path

GT = Path("/content/drive/MyDrive/pcfno3d_grand_tunis")   # code et poids du Grand Tunis
NAT = Path("/content/drive/MyDrive/pcfno3d_national")
NAT.mkdir(parents=True, exist_ok=True)
JOURNAUX = NAT / "journaux"
JOURNAUX.mkdir(exist_ok=True)
libre = shutil.disk_usage("/content/drive/MyDrive").free / 1e9
print(f"place libre sur le Drive : {libre:.0f} Go (le jeu national en demande ~50)")

CODE = Path("/content/PFE")
if CODE.exists():
    shutil.rmtree(CODE)
CODE.mkdir(parents=True)
with zipfile.ZipFile(GT / "pcfno3d_code.zip") as zf:
    zf.extractall(CODE)
tampon = CODE / "PCFNO3D_VERSION.txt"
print(tampon.read_text().strip() if tampon.exists() else "tampon : absent")
%cd /content/PFE
!pip install -q --no-deps -e . 2>&1 | tail -2
!pip install -q pandera xarray netCDF4 h5netcdf 2>&1 | tail -2
sys.path[:0] = ["/content/PFE", "/content/PFE/src"]

def lancer(nom: str, commande: str) -> None:
    """Commande obligatoire ; sortie affichee ET journalisee sur le Drive ; arret si echec."""
    t0 = time.time()
    with open(JOURNAUX / f"{nom}.log", "a", encoding="utf-8") as log:
        log.write(f"\n===== {time.strftime('%Y-%m-%d %H:%M:%S')} : {commande}\n")
        p = subprocess.Popen(commande, shell=True, cwd="/content/PFE", stdout=subprocess.PIPE,
                             stderr=subprocess.STDOUT, text=True, bufsize=1)
        for ligne in p.stdout:
            print(ligne, end="")
            log.write(ligne)
            log.flush()
        p.wait()
    print(f"[{nom}] {time.time() - t0:.0f} s, code {p.returncode}")
    if p.returncode:
        raise RuntimeError(f"{nom} a echoue (code {p.returncode}) : voir {JOURNAUX / (nom + '.log')}")

# donnees : grille nationale, grille_reelle.json, forcage ERA5 (une archive sur le Drive)
DON = Path("/content/donnees")
G = DON / "national"
if not (G / "forcage_era5.csv").exists():
    with zipfile.ZipFile(NAT / "national_donnees.zip") as zf:
        zf.extractall(DON)
print(sorted(p.name for p in G.iterdir()))

# Herbie : telechargement partiel des fichiers GFS (index .idx) ; cfgrib / eccodes : lecture GRIB2
!pip install -q herbie-data cfgrib eccodes 2>&1 | tail -1

# code du depot public au commit fige (remplace l'archive du Drive, plus ancienne)
COMMIT = "d3b2f07260a6438f70b1251be8ff35d178af6c9f"
os.chdir("/content")    # le noyau etait dans /content/PFE : le supprimer laisserait pip dans un dossier efface
subprocess.run(f"rm -rf /content/PFE && git init -q /content/PFE && cd /content/PFE && git remote add origin "
               f"https://github.com/ines-rouatbi/pcfno3d.git && git fetch -q --depth 1 origin {COMMIT} "
               f"&& git checkout -q FETCH_HEAD", shell=True, check=True)
os.chdir("/content/PFE")
r = subprocess.run([sys.executable, "-m", "pip", "install", "-q", "--no-deps", "-e", "/content/PFE"],
                   capture_output=True, text=True)
if r.returncode:
    print(r.stdout[-3000:], r.stderr[-3000:])
    raise SystemExit("pip install -e /content/PFE a echoue (sortie ci-dessus)")
import importlib
importlib.invalidate_caches()
import hashlib
for f, h in (("regle_v4_national.json", "60671e843ee648ee2422168a7659a53dfe8aa414be7b1c7c6f54eeec622a43f8"), ("regle_os7_national.json", "4360860450c4f317f365d898afdeca8e426a7a0e7523cb9d63de6da0def3bc8d")):
    assert hashlib.sha256((CODE / "configs" / "national" / f).read_bytes()).hexdigest() == h, f
print("code", COMMIT, "| regles v4 et OS7 national verifiees")

import torch
GPU = torch.cuda.is_available()
OS7N = NAT / "os7"                         # sorties de l'OS7 national (rien dans les dossiers du Grand Tunis)
OS7N.mkdir(exist_ok=True)
F = {e: OS7N / f"forcage_gfs_f{e}.csv" for e in (24, 48)}
print("environnement :", "GPU " + torch.cuda.get_device_name(0) if GPU else "CPU", "| partie", "B" if GPU else "A")

## 2. Partie A (Colab CPU) : prévisions GFS nationales, puis fin de la session

In [ ]:
# PARTIE A (Colab CPU) : previsions GFS f24 / f48 aux instants du cache de la verite v3, forcage national
from google.colab import runtime
import traceback
if GPU:
    print("GPU : la partie A (telechargement GFS) se fait en CPU ; on passe a la partie B")
else:
    try:
        # instants : l'index du cache v3 (les 3 parties), sans extraire les ~50 Go du cache
        CACHE = Path("/content/national/cache")
        CACHE.mkdir(parents=True, exist_ok=True)
        import pandas as pd
        parts = []
        for k in (1, 2, 3):
            with zipfile.ZipFile(NAT / f"cache_v3_partie_{k}.zip") as zf:
                with zf.open(f"index_partie_{k}.csv") as fi:
                    parts.append(pd.read_csv(fi))
        pd.concat(parts).drop_duplicates("stem").to_csv(CACHE / "index.csv", index=False)
        print(pd.read_csv(CACHE / "index.csv")["split"].value_counts().to_string())
        for e in (24, 48):
            if not F[e].exists():
                # boite nationale ; les echeances deja telechargees par l'OS7 du Grand Tunis sont relues sans
                # y ecrire ; code 4 = plus de 2 % d'echeances absentes de l'archive (regle) : arret
                lancer(f"gfs_national_f{e}", f"python -u -m pcfno3d.gfs --config national_1km_colab --national "
                                             f"--echeance {e} --sortie {F[e]} --reprise {OS7N / f'gfs_f{e}_tn'} "
                                             f"--lecture {GT / 'os7' / f'gfs_f{e}_instants'} --travailleurs 8")
        for e in (24, 48):
            err = F[e].with_suffix(".erreurs.json")
            print(f"f{e} :", F[e].exists(), "| echeances absentes :",
                  len(json.loads(err.read_text())) if err.exists() else 0)
        print("partie A terminee : relancer ce carnet en A100 pour la partie B")
    except Exception:
        traceback.print_exc()
    finally:
        runtime.unassign()
    raise SystemExit("fin de la partie A")

## 3. Partie B (Colab A100) : cache v3 et contrôle de la vérité

In [ ]:
# PARTIE B (Colab A100) : il faut les deux forcages GFS de la partie A
for e in (24, 48):
    if not F[e].exists():
        raise FileNotFoundError(f"{F[e]} absent : faire d'abord la partie A (Colab CPU)")
OBS = G / "obs"                             # METAR des 12 aeroports (national_donnees.zip)
if not OBS.exists():
    raise FileNotFoundError(f"{OBS} absent : les mesures des aeroports sont requises (PREV)")

LOC = Path("/content/national")
CACHE, JEU = LOC / "cache", LOC / "jeu"
for k in (1, 2, 3):
    if not (NAT / f"cache_v3_partie_{k}.zip").exists():
        raise FileNotFoundError(f"cache_v3_partie_{k}.zip absent : finir le carnet de generation")
if not (CACHE / "index.csv").exists():
    CACHE.mkdir(parents=True, exist_ok=True)
    for k in (1, 2, 3):
        with zipfile.ZipFile(NAT / f"cache_v3_partie_{k}.zip") as zf:
            zf.extractall(CACHE)
    with zipfile.ZipFile(NAT / "jeu_v3.zip") as zf:
        zf.extractall(JEU)
    import pandas as pd
    idx = pd.concat([pd.read_csv(CACHE / f"index_partie_{k}.csv") for k in (1, 2, 3)])
    idx.drop_duplicates("stem").to_csv(CACHE / "index.csv", index=False)
import pandas as pd
print(pd.read_csv(CACHE / "index.csv")["split"].value_counts().to_string())
lancer("controle", "python -u -m pcfno3d -c national_1km_colab check")

## 4. Mémoire du GPU

In [ ]:
import torch
from pcfno3d.config import Config
from pcfno3d.geometry import Geometry
from pcfno3d.model import build_model

def essai(recalcul: bool) -> float:
    cfg = Config.load("configs/national_1km_colab.yaml")
    cfg.model.recalcul_activations = recalcul
    geom = Geometry(cfg, device="cuda")
    m = build_model(cfg, geom.c_in).cuda().train()
    x = torch.randn(cfg.train.batch_size, geom.c_in, *geom.shape, device="cuda")
    torch.cuda.reset_peak_memory_stats()
    with torch.autocast("cuda", dtype=torch.bfloat16):
        m(x).float().pow(2).mean().backward()
    pic = torch.cuda.max_memory_allocated() / 1e9
    del m, x
    torch.cuda.empty_cache()
    return pic

total = torch.cuda.get_device_properties(0).total_memory / 1e9
print(torch.cuda.get_device_name(0), f"{total:.0f} Go")
try:
    pic = essai(False)
    RECALCUL = pic > 0.85 * total
except torch.cuda.OutOfMemoryError:
    torch.cuda.empty_cache()
    pic, RECALCUL = float("nan"), True
if RECALCUL:
    pic = essai(True)
print(f"pic {pic:.1f} Go sur {total:.0f} | recalcul des activations : {RECALCUL}")
OPTS = f"--set model.recalcul_activations={str(RECALCUL).lower()}"

## 5. Référence, affinage v5, PREV1-3, champ 3D, verdicts, fin de la session

In [ ]:
import shlex
def choisir_poids(src):
    """Le Drive garde parfois deux fichiers du meme nom (grand_tunis_v4_dso : deux best.pt de 1,69 Go,
    vus le 31/10) ; Colab peut n'en montrer qu'un, au hasard. On retient celui dont l'epoque est celle
    du test evalue (eval_pcfno3d/best_wcont, sinon best), meme s'il n'y en a qu'un : jamais un poids
    pris au hasard."""
    cands = sorted(src.glob("best*.pt"))
    if not cands:
        return src / "best.pt"
    attendu = next((json.loads(f.read_text()).get("epoque") for d in ("best_wcont", "best")
                    if (f := src / "eval_pcfno3d" / d / "test_summary.json").exists()), None)
    import torch
    epoques = {}
    for c in cands:
        ck = torch.load(str(c), map_location="cpu", mmap=True, weights_only=False)
        epoques[c] = ck.get("epoch", -2) + 1
        del ck
    print("poids candidats :", {c.name: e for c, e in epoques.items()}, "| epoque evaluee :", attendu)
    ok = [c for c in cands if epoques[c] == attendu] if attendu is not None else cands
    if not ok:
        raise RuntimeError(f"aucun poids de {src} n'a l'epoque evaluee ({attendu}) : verifie sur le Drive")
    return max(ok, key=lambda c: c.stat().st_mtime)          # a epoque egale : le plus recent

from google.colab import runtime
import traceback
CHOIX = NAT / "runs" / "choix_v4.json"
if not CHOIX.exists():
    raise FileNotFoundError(f"{CHOIX} absent : le carnet de test v4 choisit l'entrainement de depart")
choix = json.loads(CHOIX.read_text())
if choix.get("regle_sha256") != "60671e843ee648ee2422168a7659a53dfe8aa414be7b1c7c6f54eeec622a43f8":
    raise RuntimeError(f"choix_v4.json ne vient pas de la regle v4 verifiee ({choix.get('regle_sha256')})")
V4 = NAT / "runs" / f"national_1km_v4_{choix['entrainement_choisi']}"
V5_CONF = "national_v5_gfs_colab"
R5 = NAT / "runs" / "national_1km_v5_gfs"
EPOQUES = 12                               # configs/national_v5_gfs_colab.yaml (train.epochs)
print("depart (v4 choisi sur la validation) :", V4.name, "| run v5 :", R5)

def forcage(e):                            # le forcage GFS d'une echeance, la ou la configuration le lit
    return f"--set paths.forcage_era5={G / F[e].name}"

def tolerer_code3(nom, cmd):               # code 3 = critere non conforme : le resultat est ecrit
    try:
        lancer(nom, cmd)
    except RuntimeError as exc:
        if "(code 3)" not in str(exc):
            raise
        print(exc)

for e, f in F.items():
    for suffixe in (".csv", ".npz"):
        shutil.copy2(f.with_suffix(suffixe), G / f.with_suffix(suffixe).name)
try:
    INIT = shlex.quote(str(choisir_poids(V4)))
    # 1. reference : le v4 choisi nourri par GFS, SANS affinage (son run_dir lu seulement : statistiques)
    for e in (24, 48):
        d = OS7N / f"sans_affinage_f{e}"
        if not (d / "prevision_bilan.json").exists():
            tolerer_code3(f"prevision_sans_affinage_f{e}",
                          f"python -u scripts/evaluer_prevision.py -c {V5_CONF} {OPTS} {forcage(e)} --checkpoint {INIT} "
                          f"--set paths.run_dir={shlex.quote(str(V4))} --obs {OBS} --echeance {e} --sortie {d} "
                          f"--bootstrap 5000 --graine 2030")

    # 2. affinage v5 (GFS +24 h en entree, verite v3 inchangee, epoque choisie sur la validation)
    h = R5 / "history.json"
    if not (h.exists() and any(r.get("epoch") == EPOQUES - 1 for r in json.loads(h.read_text()))):
        lancer("entrainer_v5_national", f"python -u -m pcfno3d -c {V5_CONF} {OPTS} {forcage(24)} "
                                        f"--set train.init_from={INIT} train")

    # 3. PREV1 (+24 h), PREV2 (+48 h) aux 12 aeroports ; PREV3 : decisions de traitement de la veille
    for e in (24, 48):
        d = OS7N / f"v5_f{e}"
        if not (d / "prevision_bilan.json").exists():
            tolerer_code3(f"prevision_v5_f{e}", f"python -u scripts/evaluer_prevision.py -c {V5_CONF} {OPTS} {forcage(e)} "
                                                f"--obs {OBS} --echeance {e} --sortie {d} --bootstrap 5000 --graine 2030")
        if (d / "prevision_paires.csv").exists() and not (d / "decisions" / "decisions_agricoles.json").exists():
            lancer(f"prev3_f{e}", f"python -u scripts/verifier_decisions.py --paires {d / 'prevision_paires.csv'} "
                                  f"--sources prevision,gfs,persistance --sortie {d / 'decisions'}")

    # 4. information : le champ 3D prevu a +24 h contre la verite d'entrainement v3 (test lu une fois)
    if not (R5 / "eval_pcfno3d" / "best_wcont" / "test_summary.json").exists():
        lancer("evaluer_v5_national_f24", f"python -u -m pcfno3d -c {V5_CONF} {OPTS} {forcage(24)} evaluate")
    # (la configuration v5 lit forcage_gfs_f24.csv la ou il vient d'etre copie : la prevision a +24 h)
    if not (R5 / "dso_hauteur_sol" / "regle.json").exists():
        lancer("dso_sol_regle_v5", f"python -u scripts/dso_hauteur_sol.py -c {V5_CONF} regle")
    if not (R5 / "dso_hauteur_sol" / "bilan.json").exists():
        tolerer_code3("dso_sol_v5", f"python -u scripts/dso_hauteur_sol.py -c {V5_CONF} mesure")

    # 5. verdicts de la regle (empreinte verifiee, seuils recopies de la regle, rien de recalcule)
    lancer("juger_os7_national", f"python -u scripts/juger_os7_national.py --os7 {OS7N}")
except Exception:
    traceback.print_exc()
finally:
    runtime.unassign()